# 👗 Personalized AI Fashion Assistant
**Multimodal LangGraph agent + RAG that recommends accessories from an outfit photo, ranked as an "Influencer Suggestion".**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsyaagirl/fashion-ai-assistant/blob/main/notebooks/fashion_assistant_colab.ipynb)

This notebook runs the whole project top to bottom:

| Step | What happens | Time (free Colab, GPU) |
|---|---|---|
| 0. Setup | install packages, load your API keys | 2 min |
| 1. Data | download H&M sales data, pick 3,000 accessories, score popularity & trends | 10–15 min |
| 2. TikTok (optional) | blend in your own TikTok Shop export | 1 min |
| 3. RAG index | embed product photos with CLIP, build the style knowledge base | 3–5 min |
| 4. Agent | run the LangGraph agent on an outfit photo | ~20 s per photo |
| 5. Evaluation | compare against keyword search, write `docs/RESULTS.md` | 5 min offline + ~10 min LLM judge |
| 6. Demo | Gradio web app with a public link | – |

**Before you start:** `Runtime → Change runtime type → T4 GPU` (CLIP is much faster on GPU), and add three secrets with the 🔑 icon in the left sidebar: `GOOGLE_API_KEY` ([free key](https://aistudio.google.com/apikey)), `KAGGLE_USERNAME`, `KAGGLE_KEY` (kaggle.com → Settings → API → Create New Token). Also open the [H&M competition page](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/rules) once and click **I Understand and Accept** – downloads fail with 403 until you do.

## 0. Setup

In [ ]:
# Clone the repo from GitHub (or set USE_UPLOAD = True to upload the zip instead)
REPO_URL = "https://github.com/itsyaagirl/fashion-ai-assistant"
import os
USE_UPLOAD = False
if not os.path.exists("/content/fashion-ai-assistant"):
    if not USE_UPLOAD:
        !git clone -q $REPO_URL /content/fashion-ai-assistant
    else:
        # upload fashion-ai-assistant.zip with the file picker
        from google.colab import files
        up = files.upload()
        !unzip -q -o {list(up)[0]} -d /content/
%cd /content/fashion-ai-assistant

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os, sys
sys.path.insert(0, "src")

# Secrets: Colab sidebar 🔑 -> add GOOGLE_API_KEY, KAGGLE_USERNAME, KAGGLE_KEY
try:
    from google.colab import userdata
    for k in ["GOOGLE_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
        os.environ[k] = userdata.get(k)
except Exception as e:
    print("Not in Colab or secret missing - set environment variables yourself:", e)

# Change the model here if Google renames/retires it (see docs/HOW_WE_BUILT_IT.md)
os.environ.setdefault("GEMINI_MODEL", "gemini-3.8-flash")
os.environ.setdefault("GEMINI_JUDGE_MODEL", "gemini-3.5-flash-lite")

import pandas as pd, numpy as np
from fashion_assistant.config import settings
print("project root:", settings.project_root)
print("agent model :", settings.llm_model, "| judge:", settings.judge_model)

## 1. Data: real H&M sales → accessory catalog with popularity & trend scores

The [H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations) dataset has ~105k products with photos and **31 million real purchases** (Sep 2018 – Sep 2020). We:

1. stream the transactions once and count sales per product (all-time, last 4 weeks, the 12 weeks before);
2. **popularity score** = percentile of all-time sales; **trend score** = percentile of "how much faster is it selling now than before" (damped so tiny numbers don't count as trends);
3. keep the 3,000 best-selling accessories (product groups *Accessories* + *Bags*);
4. download only those 3,000 photos (not the 30 GB image dump);
5. build an **evaluation set** from real baskets: a garment and an accessory bought by the same customer on the same day.

In [ ]:
from fashion_assistant.data.hm_loader import prepare_all
catalog = prepare_all(with_images=True, eval_queries=200)   # re-running skips finished downloads
catalog[["prod_name", "product_type_name", "colour_group_name", "total_sales", "recent_sales", "trend_score"]].head(10)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
catalog["product_type_name"].value_counts().head(12).plot.barh(ax=ax[0], title="Catalog: top accessory types")
catalog.groupby("product_type_name")["trend_score"].mean().sort_values().tail(12).plot.barh(ax=ax[1], title="Average trend score by type")
plt.tight_layout(); plt.show()

In [ ]:
from fashion_assistant.viz import show_items
show_items(catalog.nlargest(10, "trend_score"), "Fastest-rising accessories (end of dataset)")

## 2. (Optional) TikTok Shop trend boost

There is no free public TikTok Shop dataset. If you export products yourself (e.g. an Apify TikTok Shop scraper with keyword *fashion accessories*), upload the CSV/JSON here. The loader finds which words sell (e.g. *pearl*, *claw clip*) and gives every H&M item a `tiktok_score` that is blended into its trend score. **Skip this cell if you have no export** – everything still works.

In [ ]:
USE_TIKTOK = False   # set True and run to upload your export(s)
if USE_TIKTOK:
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        (settings.data_dir / "tiktok" / name).write_bytes(data)

from fashion_assistant.data.tiktok_loader import load_tiktok_products, trending_terms, attach_tiktok_scores
terms = trending_terms(load_tiktok_products())
print(f"{len(terms)} trending TikTok terms" if terms else "No TikTok exports - tiktok_score = 0")
sorted(terms.items(), key=lambda x: -x[1])[:15]

## 3. Multimodal RAG index

* **Catalog index** – every product photo goes through **CLIP**, which puts images and text in the same vector space. Product vector = 75 % photo + 25 % short text. Stored in **FAISS**. So the agent can search with words ("thin gold hoop earrings") and the ranker can compare your outfit photo directly with product photos.
* **Style knowledge base** – `knowledge/style_guide.md` (necklines, colour rules, metals, occasions, personas) split into sections, embedded with MiniLM, stored in LangChain's `InMemoryVectorStore`.

In [ ]:
from fashion_assistant.rag.catalog_index import CatalogIndex, as_langchain_retriever
catalog = attach_tiktok_scores(pd.read_parquet(settings.processed_dir / "catalog.parquet"), terms)
index = CatalogIndex.build(catalog)
index.save()
print(index.vectors.shape)

In [ ]:
# Text -> image search works because CLIP shares one space for both
show_items(index.search_text("thin gold hoop earrings", k=5), "query: 'thin gold hoop earrings'")
show_items(index.search_text("black leather crossbody bag", k=5, category="bag"), "query: 'black leather crossbody bag'")

# The same index as a standard LangChain retriever (returns Documents)
retriever = as_langchain_retriever(index, k=3)
[d.metadata["article_id"] for d in retriever.invoke("pearl necklace")]

In [ ]:
from fashion_assistant.rag.style_kb import StyleKnowledgeBase
kb = StyleKnowledgeBase()
print(kb.n_sections, "sections")
for h in kb.search("what jewellery goes with a v-neck black dress for a wedding"):
    print(f"[{h['section']}] {h['text'][:140]}...")

## 4. The LangGraph agent

```
analyze_outfit ─(not clothing)→ not_an_outfit → END
      │
    agent ⇄ tools        (Gemini decides which searches / rule look-ups to run; max 4 rounds)
      │
collect_candidates → rank (Influencer score × personas) → write_up → END
```
* `analyze_outfit` – Gemini vision returns a typed `OutfitAnalysis` (garments, colours, style, occasion, missing categories).
* `agent` + `tools` – Gemini with three tools bound: `search_accessories`, `get_trending_accessories`, `lookup_style_rules`.
* `rank` – deterministic **Influencer score** = style fit (CLIP) + trend fit (sales) + colour harmony (+ persona taste). Run once for the balanced "editor" pick and once per persona.
* `write_up` – Gemini explains the picks in each influencer's voice.

In [ ]:
from fashion_assistant import FashionAssistant
fa = FashionAssistant(index=index, kb=kb)
try:
    from IPython.display import Image as IPImage, display
    display(IPImage(fa.graph.get_graph().draw_mermaid_png()))
except Exception as e:      # draw_mermaid_png calls an online renderer; fall back to text
    print(fa.mermaid())

In [ ]:
# Use your own outfit photo, or fall back to a garment from the eval set
USE_MY_PHOTO = False
if USE_MY_PHOTO:
    from google.colab import files
    up = files.upload(); photo = list(up)[0]
else:
    from fashion_assistant.data.hm_loader import local_image_path
    ev = pd.read_parquet(settings.processed_dir / "eval_set.parquet")
    photo = str(local_image_path(ev.iloc[0]["garment_id"]))
    print("outfit:", ev.iloc[0]["garment_text"][:120])

from PIL import Image
display(Image.open(photo).resize((256, 384)))

In [ ]:
result = fa.recommend(photo, request="going out for dinner", personas=["minimalist", "streetwear", "luxe", "boho"])

from fashion_assistant.viz import print_trace
print("Outfit analysis:", result["outfit"]); print()
print_trace(result)

In [ ]:
from IPython.display import Markdown
show_items(result["ranked"], "⭐ Influencer Suggestion (editor's ranking)")
for persona, picks in result["persona_picks"].items():
    show_items(picks, f"{persona} picks", cols=2)
Markdown(result["final_answer"])

In [ ]:
# Why did #1 win? The evidence behind each score
pd.DataFrame(result["ranked"])[["prod_name", "style_fit", "trend_fit", "color_fit", "influencer_score", "evidence"]]

## 5. Evaluation: side by side against keyword search

**Part 1 – offline (free, all ~200 outfits).** Ground truth = accessories really bought together with the garment. Four systems:
* **Keyword search (BM25)** – what a normal shop search box does with the garment's text
* **Popularity** – best sellers for everyone
* **CLIP retrieval** – our RAG step alone (photo → photos)
* **Influencer ranker (ours)** – CLIP retrieval + style/trend/colour re-ranking

**Part 2 – blind LLM judge (20 outfits).** The full agent's top 5 vs keyword search's top 5, shown to a separate Gemini judge in random order with the outfit photo.

> Bought-together data is noisy (people buy unrelated things in one trip), so absolute numbers are small for every system; what matters is the *relative* comparison.

In [ ]:
from fashion_assistant.eval.run_eval import run_offline_eval, run_judge_eval, save_results
ev = pd.read_parquet(settings.processed_dir / "eval_set.parquet")
table, systems = run_offline_eval(index, ev, k=10)
table

In [ ]:
JUDGE_N = 20       # each outfit = ~4 Gemini calls for the agent + 1 for the judge
judge = run_judge_eval(fa, ev, n=JUDGE_N)
judge

In [ ]:
save_results(table, judge, len(ev))
Markdown((settings.docs_dir / "RESULTS.md").read_text())

## 6. Demo app

Launches a Gradio UI with a public link (valid ~72 h). Stop the cell to shut it down.

In [ ]:
!python app.py --share

## 7. Tests & saving your work

In [ ]:
!python -m pytest -q tests

In [ ]:
# Commit the generated results (data/ is git-ignored on purpose)
# !git add docs/RESULTS.md docs/results.json docs/results_offline.csv docs/images/offline_metrics.png
# !git commit -m "Add evaluation results" && git push